<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w12-dsa-for-agents/notebook.ipynb)


In [39]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [40]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w12-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 12: Data structures for agents

**Week 0 · Course C, one unit · about 60 minutes**

**Goal:** Measure a lookup before you trust it, pick list, dict or set on purpose, give every walk a budget that stops it, and turn "derive X before Y" edges into an order, or into a gap.

**Why it matters:** The agent loop in session 5 is a queue with a budget. The corpus index and the evaluator are dicts keyed by id. Session 13 says "callable is a graph property". Every one of those is a data structure, and this unit is where you meet them on the course's own data.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and
read what happens. Debugging something wrong teaches more than filling in a blank.

## 1. Measure the lookup

**Context.** Six documents in `data/corpus/`; a scan over them is instant. Sixty thousand is
another world, and the only way to know which world you are in is to measure. `timeit.repeat`
runs a statement many times and returns one figure per repeat. The minimum is the honest one:
every slower run was the same code plus noise. This cell scales the six doc ids to 60,000 and
looks up the last one twice: a linear scan, then a dict lookup.

**Instructions.**

1. Run the cell. The dict lookup comes out slower than the scan. Read `lookup()`, the function
   timeit repeats.
2. Building a 60,000-entry dict is work, and it happens on every repetition. Build `index` once,
   outside `lookup()`, and leave only the lookup inside.
3. Leave the label as `"measured"`. It is a promise that the numbers came from timeit, not from
   memory.

**Expected output** (the numbers are yours; the ratio is in the thousands)

```
6 documents, scaled to 60000 ids; target = 'structured-outputs-9999'
linear scan: 0.561 ms    dict lookup: 0.000032 ms    ratio: 17,780x
✅ w12-e1 passed
```

In [44]:
import timeit

from bootcamp_agent.documents import load_corpus

docs = load_corpus(REPO_ROOT / "data" / "corpus")
doc_ids = [f"{doc.doc_id}-{copy}" for copy in range(10_000) for doc in docs]
target = doc_ids[-1]
print(f"{len(docs)} documents, scaled to {len(doc_ids)} ids; target = {target!r}")


def scan(wanted):
    """Look at every id until one matches: O(n)."""
    for position, doc_id in enumerate(doc_ids):
        if doc_id == wanted:
            return position
    return None

# 1. Construire l'index UNE SEULE FOIS en dehors de lookup() et du timeit
index = {doc_id: position for position, doc_id in enumerate(doc_ids)}

def lookup():
    """The statement timeit repeats. Only the lookup belongs in here."""
    return index[target]


REPEAT, NUMBER = 5, 20
linear_ms = min(timeit.repeat(lambda: scan(target), number=NUMBER, repeat=REPEAT)) / NUMBER * 1000
dict_ms = min(timeit.repeat(lookup, number=NUMBER, repeat=REPEAT)) / NUMBER * 1000
timings = {"linear_ms": linear_ms, "dict_ms": dict_ms, "label": "measured"}
print(f"linear scan: {linear_ms:.3f} ms    dict lookup: {dict_ms:.6f} ms    ratio: {linear_ms / dict_ms:,.0f}x")

6 documents, scaled to 60000 ids; target = 'structured-outputs-9999'
linear scan: 1.027 ms    dict lookup: 0.000031 ms    ratio: 32,751x


In [45]:
check("w12-e1", timings)

✅ w12-e1 passed


True

## 2. One of each, in the order seen

**Context.** The agent cites doc ids, and it cites the same one twice when two chunks of one
document are retrieved. The answer should list each source once, in the order the agent first
used it. A `set` gives one of each and forgets the order. A `dict` gives one of each key and,
since Python 3.7, keeps the order; `dict.fromkeys` builds one from any iterable. The same shape
is everywhere in the repo: the corpus indexed by doc id, the golden set keyed by question, a
chunk keyed by the tuple `(doc_id, position)`.

**Instructions.**

1. Run the cell. The count of sources is right and the order is whatever the hashes decided.
   Restart the kernel and run again: it can change.
2. Rewrite `dedupe` so the order is first-seen: `dict.fromkeys`, or a seen-set beside a list.
3. Do not sort. Sorting answers a different question.

**Expected output**

```
index: 6 documents by doc id; by_id['rag-basics'].title = 'RAG Basics'
golden: 8 questions, 8 distinct; 23 chunks keyed by (doc_id, position)
cited: ['rag-basics', 'agent-loops', 'rag-basics', 'mcp-overview', 'agent-loops', 'rag-basics']
sources: ['rag-basics', 'agent-loops', 'mcp-overview']
✅ w12-e2 passed
```

In [54]:
import json


from bootcamp_agent.documents import load_corpus

from bootcamp_agent.retrieval import chunk_document


docs = load_corpus(REPO_ROOT / "data" / "corpus")

by_id = {doc.doc_id: doc for doc in docs}  # dict: one entry per id, found in O(1)

rows = [json.loads(line) for line in (REPO_ROOT / "data" / "evals" / "golden.jsonl").read_text().splitlines()]

golden = {row["question"]: row for row in rows}  # dict: the eval set, keyed by question

chunks = [chunk for doc in docs for chunk in chunk_document(doc)]

by_chunk = {(chunk.doc_id, chunk.position): chunk for chunk in chunks}  # a tuple is hashable: a key


citations = ["rag-basics", "agent-loops", "rag-basics", "mcp-overview", "agent-loops", "rag-basics"]



def dedupe(citations):
    """One entry per doc id, in the order the agent first cited it."""
    return list(dict.fromkeys(citations))  # <------ EDIT THIS LINE


print(f"index: {len(by_id)} documents by doc id; by_id['rag-basics'].title = {by_id['rag-basics'].title!r}")

print(f"golden: {len(rows)} questions, {len(golden)} distinct; {len(by_chunk)} chunks keyed by (doc_id, position)")

print("cited:", citations)


index: 6 documents by doc id; by_id['rag-basics'].title = 'RAG Basics'
golden: 8 questions, 8 distinct; 23 chunks keyed by (doc_id, position)
cited: ['rag-basics', 'agent-loops', 'rag-basics', 'mcp-overview', 'agent-loops', 'rag-basics']


In [55]:
check("w12-e2", dedupe)

✅ w12-e2 passed


True

## 3. A walk with a budget

**Context.** An agent run is a tree of tool calls: the question at the root, each search a child,
each document read a grandchild. Walking it is recursion: visit the node, then walk each child.
The loop-engineering guide says stopping conditions come first, and a budget is one: the walk
visits at most `budget` names, then returns, whatever the tree looks like. A tree that refers to
itself is not broken data; it is a tool that calls the tool that called it, and the budget is
the only thing that ends it.

**Instructions.**

1. Run the cell. The walk visits all seven names and ignores `budget=3`.
2. Check the budget before the work: when `len(visited)` has reached `budget`, return.
3. The checker runs your walk on a node whose children contain itself, under a deadline. Without
   the check, the interpreter stops it with `RecursionError`; with it, five names come back.

**Expected output**

```
budget 3: ['answer: how does chunking work?', "search('chunking')", "get_document('rag-basics')"]
budget 10: 7 names
✅ w12-e3 passed
```

In [56]:
tool_calls = {
    "name": "answer: how does chunking work?",
    "children": [
        {"name": "search('chunking')", "children": [
            {"name": "get_document('rag-basics')", "children": []},
            {"name": "get_document('evaluation-basics')", "children": []},
        ]},
        {"name": "search('retrieval')", "children": [
            {"name": "get_document('rag-basics')", "children": []},
            {"name": "get_document('mcp-overview')", "children": []},
        ]},
    ],
}


def walk(node, budget, visited=None):
    """Visit names depth-first, at most `budget` of them."""
    visited = [] if visited is None else visited
    if len(visited) >= budget:  # <------ EDIT THIS LINE
        return visited
    visited.append(node["name"])
    for child in node["children"]:
        walk(child, budget, visited)
    return visited

print("budget 3:", walk(tool_calls, budget=3))
print("budget 10:", len(walk(tool_calls, budget=10)), "names")

budget 3: ['answer: how does chunking work?', "search('chunking')", "get_document('rag-basics')"]
budget 10: 7 names


In [57]:
check("w12-e3", walk)

✅ w12-e3 passed


True

## 4. Derive order from edges

**Context.** `cookbook/fixtures/program-graph-example.json` is an instruction-to-account graph in
the shape Gecko produces for a small storefront program. Every account derived from another
account is an edge: derive `authority` before `store`, `store` before `item`, `mint` before
`buyer_token_account`. An order that respects every edge is a topological order, and the way to
find one is a queue of ready nodes: place a node only when everything it needs has been placed.
When nodes remain and none is ready, the edges form a cycle. A cycle is a gap, not an order.

**Instructions.**

1. Run the cell. The edges print, then an "order" that is only the order the names were first
   seen in: `mint` lands after `buyer_token_account`, which needs it.
2. Rewrite `derive_order`: count incoming edges, queue the nodes with zero, pop one, place it,
   lower the counts of what it points to. `collections.deque` is the queue.
3. Return `None` when the queue runs dry with nodes left. Do not raise, do not loop.

**Expected output**

```
derive 'authority' before 'store'
derive 'store' before 'item'
derive 'item_index' before 'item'
derive 'buyer' before 'buyer_token_account'
derive 'mint' before 'buyer_token_account'
derive 'item' before 'mint'
order: ['authority', 'item_index', 'buyer', 'store', 'item', 'mint', 'buyer_token_account']
cycle: None
✅ w12-e4 passed
```

In [58]:
import json
from collections import deque

fixture = json.loads((REPO_ROOT / "cookbook" / "fixtures" / "program-graph-example.json").read_text())

edges = []
for instruction in fixture["instructions"]:
    for account in instruction["accounts"]:
        for seed in account.get("seeds", []):
            if not seed.startswith("'"):  # a quoted seed is a constant; nothing derives it
                edges.append((seed, account["name"]))
        for source in account.get("derived_from", []):
            edges.append((source, account["name"]))
        if "read_from" in account:
            edges.append((account["read_from"], account["name"]))
edges = list(dict.fromkeys(edges))  # exercise 2 again: one of each, first seen
for before, after in edges:
    print(f"derive {before!r} before {after!r}")


def derive_order(edges):
    """An order that respects every (before, after) edge, or None when there is a cycle."""
    nodes = list(dict.fromkeys(node for edge in edges for node in edge))
    
    # Construction du graphe et calcul des degrés entrants
    in_degree = {node: 0 for node in nodes}
    graph = {node: [] for node in nodes}
    
    for before, after in edges:
        graph[before].append(after)
        in_degree[after] += 1

    # File d'attente des nœuds prêts (degré entrant égal à 0)
    queue = deque([node for node in nodes if in_degree[node] == 0])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                queue.append(neighbor)

    # S'il reste des nœuds, c'est qu'il y a un cycle
    if len(order) != len(nodes):
        return None

    return order


print("order:", derive_order(edges))
print("cycle:", derive_order([("store", "item"), ("item", "mint"), ("mint", "store")]))
check("w12-e4", derive_order)

derive 'authority' before 'store'
derive 'store' before 'item'
derive 'item_index' before 'item'
derive 'buyer' before 'buyer_token_account'
derive 'mint' before 'buyer_token_account'
derive 'item' before 'mint'
order: ['authority', 'item_index', 'buyer', 'store', 'item', 'mint', 'buyer_token_account']
cycle: None
✅ w12-e4 passed


True

In [59]:
check("w12-e4", derive_order)

✅ w12-e4 passed


True

## Review

The scorecard for this unit. Every ❌ line names the exercise and the fix.

In [60]:
review("w12")

w12: 4/4 passed  ·  400/400 marks


True